<a href="https://colab.research.google.com/github/metalBigLesson/PySolitaire/blob/main/Solitaire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from enum import IntEnum
import random
import traceback

class Suit(IntEnum):
    NONE = 0
    CLUBS = 1
    DIAMONDS = 2
    SPADES = 3
    HEARTS = 4

class Color(IntEnum):
    NONE = 0
    RED = -1
    BLACK = 1

def color(suit):
    if (suit == Suit.NONE):
        return Color.NONE
    if (suit % 2 == 0): return Color.RED
    return Color.BLACK

class Card:
    def __init__(self, suit, rank, show=True):
        self.suit = suit
        self.rank = rank
        self.show = show

    def __eq__(self, other):
        return self.suit == other.suit and self.rank == other.rank

    def __sum__(self, other):
        return self.rank + other.rank

    def __sub__(self, other):
        return self.rank - other.rank

    def __mul__(self, other):
        return color(self.suit) * color(other.suit)

    def __mod__(self, other):
        return self.suit - other.suit

    def __str__(self):
        if not self.show:
            return "[▒▒]"
        return '[' + self._suit_chr() + self._rank_str() + ']'

    def _suit_chr(self):
        match self.suit:
            case Suit.CLUBS:
                return "♣"
            case Suit.DIAMONDS:
                return "◇"
            case Suit.HEARTS:
                return "♡"
            case Suit.SPADES:
                return "♠"
            case _:
                return "　"

    def _rank_str(self):
        match self.rank:
            case 0:
                return "  "
            case 1:
                return "Ａ"
            case 10:
                return "10"
            case 11:
                return "Ｊ"
            case 12:
                return "Ｑ"
            case 13:
                return "Ｋ"
            case _:
                return chr(ord('０') + self.rank)


class Stack():
    def __init__(self, card_list=[]):
        self.cards = card_list
        for i in range(len(self.cards) - 1):
            self.cards[i].show = False
        self.selected = []

    def __str__(self):
        stack_str = ''
        for card in self.cards:
            stack_str += card.__str__()[:-1]
        if stack_str != '':
            stack_str += ']'
        selected_str = ' '
        for card in self.selected:
            selected_str += card.__str__()[:-1]
        if selected_str != ' ':
            selected_str += ']'
        return stack_str + selected_str

    def open(self):
        if len(self.cards) != 0 and len(self.selected) == 0:
            self.cards[-1].show = True

    def length(self):
        return len(self.cards)

    def is_legal(self, card1, card2): return True

    def is_empty(self):
        return len(self.cards) == 0

    def top(self):
        if (self.length() == 0):
            return Card(Suit.NONE, 0)
        return self.cards[-1]

    def push(self, cards):
        self.cards.extend(cards)
        cards.clear()

    def pop(self, index):
        if index < 0 or index >= len(self.cards): return []
        self.selected = self.cards[index:]
        self.cards = self.cards[:index]
        return self.selected

    def push_if_can(self, cards):
        if len(cards) == 0: return
        if self.is_legal(self.top(), cards[0]):
            self.push(cards)

    def pop_if_can(self, index):
        for i in range(index, len(self.cards) - 1):
            if not self.is_legal(self.cards[i], self.cards[i+1]):
                return []
        return self.pop(index)

    def on_click(self, index, hand):
        if len(self.selected) > 0:
            #print('cancel')
            self.push(self.selected)
            return []
        if len(hand) == 0:
            #print('select')
            return self.pop_if_can(index)
        #print('move')
        self.push_if_can(hand)
        return hand

class Home():
    def __init__(self, suit):
        self.cards = [Card(suit, 0)]
        self.selected = []

    def __str__(self):
        return self.cards[-1].__str__()

    def top(self):
        if (len(self.cards) == 0):
            return Card(Suit.NONE, 0)
        return self.cards[-1]

    def is_legal(self, card1, card2):
        return card1 % card2 == 0 and card2 - card1 == 1

    def push_if_can(self, cards):
        if len(cards) != 1: return
        if self.is_legal(self.top(), cards[0]):
            self.cards.append(cards[0])
            cards.clear()

    def pop_if_can(self):
        if len(self.cards) <= 1: return []
        self.selected = [self.cards.pop()]
        return self.selected

    def on_click(self, hand):
        if len(self.selected) > 0:
            self.cards.append(self.selected.pop())
            return []
        if len(hand) == 0:
            return self.pop_if_can()
        self.push_if_can(hand)
        return hand

class Deck():
    def __init__(self, card_list, opening_count, show_count):
        super().__init__()
        self.closed = card_list
        for card in self.closed:
            card.show = False
        self.opened = []
        self.opening_count = opening_count
        self.show_count = show_count
        self.selected = []

    def __str__(self):
        closed_str = '  ○  '
        if (len(self.closed) > 0):
            closed_str = self.closed[-1].__str__()
        opened_show = self.opened[-self.show_count:]
        opened_str = ''
        for card in opened_show:
            opened_str += card.__str__()[:-1]
        if opened_str != '':
            opened_str += ']'
        selected_str = ' '
        if (len(self.selected) > 0):
            selected_str += self.selected[0].__str__()
        return closed_str + opened_str + selected_str

    def open(self):
        if len(self.closed) == 0:
            for i in range(len(self.opened)):
                card = self.opened.pop()
                card.show = False
                self.closed.append(card)
        else:
            for i in range(self.opening_count):
                if len(self.closed) == 0: break
                card = self.closed.pop()
                card.show = True
                self.opened.append(card)

    def on_click(self, hand):
        if len(self.selected) > 0: #cancel
            self.opened.append(self.selected.pop())
            return []
        elif len(hand) != 0: #cannot push
            return hand
        else: #select
            if (len(self.opened) == 0):
                return []
            self.selected = [self.opened.pop()]
            return self.selected

def make_cards(is_shuffle = True):
    deck = []
    for suit in Suit:
        if suit == Suit.NONE:
            continue
        for rank in range(1, 13):
            deck.append(Card(suit, rank))
    if is_shuffle:
        random.shuffle(deck)
    return deck

In [19]:
class Klondike_Stack(Stack):
    def __init__(self, card_list=[]):
        super().__init__(card_list)

    def is_legal(self, card1, card2):
        if card1.suit == Suit.NONE:
            return card2.rank == 13
        if not card2.show:
            return False
        return card1 * card2 == -1 and card1 - card2 == 1



In [21]:
cards = make_cards()
stacks = []
temp_cards = []
for i in range(7):
    for j in range(i + 1):
        temp_cards.append(cards.pop())
    stacks.append(Klondike_Stack(temp_cards))
    temp_cards = []

deck = Deck(cards, 3, 3)
homes = [Home(Suit.CLUBS), Home(Suit.DIAMONDS), Home(Suit.SPADES), Home(Suit.HEARTS)]
hand = []

def print_game():
    print('D ', deck, sep='')
    print('H ', homes[0], homes[1], homes[2], homes[3], sep='')
    for i in range(7):
        stacks[i].open()
        print(i, ' ', stacks[i], sep='')
    print('  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12')
    print('선택된 카드: ', '없음' if len(hand) == 0 else hand[0], '...' if len(hand) > 1 else '', sep='')
    print('x, y 자리에는 정수를 입력한다.')
    print('o: 덱에서 카드 3장을 내린다.')
    print('d: 덱의 맨 위 카드를 선택한다.')
    print('hx: 왼쪽에서 x번째 홈의 카드를 선택한다.')
    print('h: 선택된 카드를 홈으로 옮긴다.')
    print('yx: 위에서 y번째 스택의, 왼쪽에서 x번째 카드부터 맨 오른쪽 카드까지 선택한다. (x의 기본값: 0)')
    print('선택된 카드가 있는 덱, 홈, 스택을 선택시 선택을 취소한다.')
    print('Q: 게임을 종료한다.')

print_game()


while True:
    try:
        input_str = input()
        if (input_str == 'Q'):
            break
        rear_str = input_str[1:]
        match input_str[0]:
            case 'o':
                deck.open()
            case 'd':
                print('1')
                hand = deck.on_click(hand)
                print('2')
            case 'h':
                home_index = int(rear_str) if rear_str != '' else int(hand[0].suit) - 1
                hand = homes[home_index].on_click(hand)
            case _:
                y = int(input_str[0])
                x = int(rear_str) if rear_str != '' else 0
                hand = stacks[y].on_click(x, hand)
        print_game()
    except Exception as e:
        print('잘못된 입력입니다.', e)
        traceback.print_exc()
        continue



D [▒▒] 
H [♣  ][◇  ][♠  ][♡  ]
0 [♠４] 
1 [▒▒[♠６] 
2 [▒▒[▒▒[♣Ａ] 
3 [▒▒[▒▒[▒▒[♡７] 
4 [▒▒[▒▒[▒▒[▒▒[♠８] 
5 [▒▒[▒▒[▒▒[▒▒[▒▒[♡５] 
6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♡３] 
  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12
선택된 카드: 없음
x, y 자리에는 정수를 입력한다.
o: 덱에서 카드 3장을 내린다.
d: 덱의 맨 위 카드를 선택한다.
hx: 왼쪽에서 x번째 홈의 카드를 선택한다.
h: 선택된 카드를 홈으로 옮긴다.
yx: 위에서 y번째 스택의, 왼쪽에서 x번째 카드부터 맨 오른쪽 카드까지 선택한다. (x의 기본값: 0)
선택된 카드가 있는 덱, 홈, 스택을 선택시 선택을 취소한다.
Q: 게임을 종료한다.
22
D [▒▒] 
H [♣  ][◇  ][♠  ][♡  ]
0 [♠４] 
1 [▒▒[♠６] 
2 [▒▒[▒▒] [♣Ａ]
3 [▒▒[▒▒[▒▒[♡７] 
4 [▒▒[▒▒[▒▒[▒▒[♠８] 
5 [▒▒[▒▒[▒▒[▒▒[▒▒[♡５] 
6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♡３] 
  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12
선택된 카드: [♣Ａ]
x, y 자리에는 정수를 입력한다.
o: 덱에서 카드 3장을 내린다.
d: 덱의 맨 위 카드를 선택한다.
hx: 왼쪽에서 x번째 홈의 카드를 선택한다.
h: 선택된 카드를 홈으로 옮긴다.
yx: 위에서 y번째 스택의, 왼쪽에서 x번째 카드부터 맨 오른쪽 카드까지 선택한다. (x의 기본값: 0)
선택된 카드가 있는 덱, 홈, 스택을 선택시 선택을 취소한다.
Q: 게임을 종료한다.
h
D [▒▒] 
H [♣Ａ][◇  ][♠  ][♡  ]
0 [♠４] 
1 [▒▒[♠６] 
2 [▒▒[♡Ｑ] 
3 [▒▒[▒▒[▒▒[♡７] 
4 [▒▒[▒▒[▒▒[▒▒[♠８] 
5 [▒▒[▒▒[▒▒[▒▒[▒▒[♡５] 
6 [▒▒[▒▒[▒▒[▒▒

D [▒▒][♣Ｋ[♠Ｋ[♡２]

H [♡  ][◇  ][♣  ][♠  ]

0 [♠Ｊ]

1 [▒▒[♣５]

2 [▒▒[▒▒[♠Ｑ]

3 [▒▒[▒▒[▒▒[◇Ａ]

4 [▒▒[▒▒[▒▒[▒▒[♠10]

5 [▒▒[▒▒[▒▒[▒▒[▒▒] [◇３]

6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♣４]

　0　　1　　2　　3　　4　　5　　6